In [1]:
# test entropy weighting
import sys
sys.path.append("../src")

import pandas as pd
from data_quality import run_quality_checks
from kalman_filter import run_kalman_stage
from entropy_weighting import make_indicators, normalize, entropy_weights

raw = pd.read_csv("../data/member1_raw_sensor_stream.csv")
df = make_indicators(run_kalman_stage(run_quality_checks(raw)))

norm = normalize(df)
weights = entropy_weights(norm)
weights.round(3)

power_ratio      0.453
voltage_ratio    0.195
current_ratio    0.227
temp_rise        0.075
dust_level       0.051
dtype: float64

In [2]:
# simple weighted score and compare with the reference
score = (norm * weights).sum(axis=1) * 100

ref = pd.read_csv("../data/member1_sensor_health.csv")
m = df.assign(score=score).merge(ref[["record_id", "run_id", "health_score"]], on="record_id")
m["fault"] = m["run_id"].str.split("-").str[2]

m.groupby("fault")[["score", "health_score"]].mean().round(1)

,score,health_score
fault,,
DUS,67.8,66.5
NOR,87.9,90.1
OPE,28.7,0.5
PAR,61.1,53.6
SHO,32.1,1.6
